In [ ]:
# In[1]:

import pandas as pd
import numpy as np
import pytz
from pathlib import Path

tz = pytz.timezone('Asia/Shanghai')

# Load files
metric_fp = Path('metric.csv')
trace_fp = Path('trace.csv')
log_fp = Path('log.csv')
error_fp = Path('error_logs.csv')

df_metric = pd.read_csv(metric_fp) if metric_fp.exists() else pd.DataFrame(columns=['timestamp','cmdb_id','kpi_name','value'])
df_trace = pd.read_csv(trace_fp) if trace_fp.exists() else pd.DataFrame(columns=['timestamp','cmdb_id','trace_name','value'])
df_log = pd.read_csv(log_fp) if log_fp.exists() else pd.DataFrame(columns=['timestamp','cmdb_id','log_name','value'])
df_error = pd.read_csv(error_fp) if error_fp.exists() else pd.DataFrame(columns=['timestamp','cmdb_id','message'])

# Normalize column names and build unified table (use full series)
df_m = df_metric.rename(columns={'kpi_name':'kpi'})[['timestamp','cmdb_id','kpi','value']].assign(source='metric')
df_t = df_trace.rename(columns={'trace_name':'kpi'})[['timestamp','cmdb_id','kpi','value']].assign(source='trace')
df_l = df_log.rename(columns={'log_name':'kpi'})[['timestamp','cmdb_id','kpi','value']].assign(source='log')

# Ensure consistent dtypes
for d in (df_m, df_t, df_l):
    if 'timestamp' in d.columns:
        d['timestamp'] = pd.to_numeric(d['timestamp'], errors='coerce').astype('Int64')
    if 'value' in d.columns:
        d['value'] = pd.to_numeric(d['value'], errors='coerce')

unified = pd.concat([df_m, df_t, df_l], ignore_index=True, sort=False)

# 1) Aggregates per (source, cmdb_id, kpi) using full series
# Compute count, min, p50, p95, max. Count counts non-null values.
grp = unified.groupby(['source','cmdb_id','kpi'], dropna=False)['value']
agg = grp.agg(
    count = lambda x: int(x.count()),
    min = lambda x: float(x.min()) if x.count()>0 else np.nan,
    p50 = lambda x: float(x.quantile(0.5)) if x.count()>0 else np.nan,
    p95 = lambda x: float(x.quantile(0.95)) if x.count()>0 else np.nan,
    max = lambda x: float(x.max()) if x.count()>0 else np.nan
).reset_index()

agg_top50 = agg.sort_values('count', ascending=False).head(50)

# 2) Overall timestamp range (min, max) for each file, displayed in Asia/Shanghai (UTC+8)
def ts_range(df, ts_col='timestamp'):
    if df is None or ts_col not in df.columns or df[ts_col].dropna().empty:
        return pd.Series({'min_ts_utc8': pd.NaT, 'max_ts_utc8': pd.NaT})
    mins = int(pd.to_numeric(df[ts_col], errors='coerce').min())
    maxs = int(pd.to_numeric(df[ts_col], errors='coerce').max())
    min_dt = pd.to_datetime(mins, unit='s', utc=True).tz_convert(tz)
    max_dt = pd.to_datetime(maxs, unit='s', utc=True).tz_convert(tz)
    return pd.Series({'min_ts_utc8': min_dt, 'max_ts_utc8': max_dt})

ts_ranges = pd.DataFrame({
    'file': ['metric','trace','log','error_logs'],
    'min_ts_utc8': [ts_range(df_metric).min_ts_utc8, ts_range(df_trace).min_ts_utc8, ts_range(df_log).min_ts_utc8, ts_range(df_error).min_ts_utc8],
    'max_ts_utc8': [ts_range(df_metric).max_ts_utc8, ts_range(df_trace).max_ts_utc8, ts_range(df_log).max_ts_utc8, ts_range(df_error).max_ts_utc8]
})

# 3) Unique cmdb_id values across all files and number of distinct kpis per cmdb_id (top 20)
cmdb_ids_all = pd.Series(pd.concat([df_metric[['cmdb_id']], df_trace[['cmdb_id']], df_log[['cmdb_id']], df_error[['cmdb_id']]], ignore_index=True)['cmdb_id'].dropna().unique())
# Count distinct kpis per cmdb_id from unified; error_logs contribute zero kpis if only present there
kpi_counts = unified.groupby('cmdb_id', dropna=False)['kpi'].nunique().rename('distinct_kpis').reset_index()
# Ensure cmdb_ids present only in error_logs are included with 0
error_only = set(df_error['cmdb_id'].dropna().unique()) - set(kpi_counts['cmdb_id'].dropna().unique())
if error_only:
    kpi_counts = pd.concat([kpi_counts, pd.DataFrame({'cmdb_id':list(error_only),'distinct_kpis':[0]*len(error_only)})], ignore_index=True, sort=False)

kpi_counts['distinct_kpis'] = kpi_counts['distinct_kpis'].astype(int)
kpi_counts_top20 = kpi_counts.sort_values('distinct_kpis', ascending=False).head(20).reset_index(drop=True)

# Compact outputs: top 50 series, timestamp ranges per file, unique cmdb_id count and top20 distinct kpis
unique_cmdb_count = int(cmdb_ids_all.size)

agg_top50, ts_ranges, unique_cmdb_count, kpi_counts_top20

```
Out[1]:
```
# Build a concise plain-English summary using the previously computed results (reuse variables from prior run)
# Assumes agg_top50, ts_ranges, unique_cmdb_count, kpi_counts_top20 exist in the kernel state.

# Prepare compact top-5 series summary (from agg_top50)
top50 = agg_top50.copy()
top5 = top50.head(5)[['source','cmdb_id','kpi','count','min','p50','p95','max']].reset_index(drop=True)

# Count how many of top50 share the maximum count (usually highest-frequency series)
max_count = int(top50['count'].max()) if not top50.empty else 0
num_with_max = int((top50['count'] == max_count).sum()) if not top50.empty else 0

# Format timestamp ranges per file into short strings
ts_map = {}
for _, row in ts_ranges.iterrows():
    f = row['file']
    mn = row['min_ts_utc8']
    mx = row['max_ts_utc8']
    ts_map[f] = (str(mn), str(mx))

# Compose summary string
summary = (
    f"Summary of telemetry aggregates (full series):\n"
    f"- Unique cmdb_id count across all files: {unique_cmdb_count}\n"
    f"- Timestamp ranges (all times shown in UTC+8):\n"
    f"  * metric:   {ts_map.get('metric',('NA','NA'))[0]}  to  {ts_map.get('metric',('NA','NA'))[1]}\n"
    f"  * trace:    {ts_map.get('trace', ('NA','NA'))[0]}  to  {ts_map.get('trace',('NA','NA'))[1]}\n"
    f"  * log:      {ts_map.get('log',   ('NA','NA'))[0]}  to  {ts_map.get('log',   ('NA','NA'))[1]}\n"
    f"  * error_logs:{ts_map.get('error_logs',('NA','NA'))[0]}  to  {ts_map.get('error_logs',('NA','NA'))[1]}\n"
    f"- Aggregates computed per (source, cmdb_id, kpi): count, min, P50, P95, max. \n"
    f"  * Returned top 50 series sorted by count desc. Maximum series count in that list = {max_count} "
    f"(#series having that count = {num_with_max}).\n"
    f"- Top 5 series (source, cmdb_id, kpi, count, min, p50, p95, max):\n"
)

# Append top5 lines succinctly
for i, r in top5.iterrows():
    summary += (
        f"  {i+1}) {r['source']} | {r['cmdb_id']} | {r['kpi']} | count={int(r['count'])} "
        f"| min={r['min']:.6g} p50={r['p50']:.6g} p95={r['p95']:.6g} max={r['max']:.6g}\n"
    )

# Add a note about distinct kpi counts (top 5 of top20 shown)
summary += "- Top cmdb_ids by number of distinct kpis (top 5 shown):\n"
for i, row in kpi_counts_top20.head(5).iterrows():
    summary += f"  {i+1}) {row['cmdb_id']} — {int(row['distinct_kpis'])} distinct kpis\n"

# Display the summary string and a compact DataFrame of the top5 series for quick reference
summary, top5

The original code execution output of IPython Kernel is also provided below for reference:

(     source                  cmdb_id                                  kpi  count         min          p50          p95          max
6072  trace        shippingservice-1  trace.from_frontend-1.duration_mean     30    0.000057     0.000060     0.000065     0.000071
6073  trace        shippingservice-1   trace.from_frontend-1.duration_p95     30    0.000058     0.000065     0.000081     0.000084
6074  trace        shippingservice-1     trace.from_frontend-1.error_rate     30    0.000000     0.000000     0.000000     0.000000
6075  trace        shippingservice-1      trace.from_frontend-1.row_count     30    3.000000     6.000000     8.550000     9.000000
6076  trace        shippingservice-1  trace.from_frontend-2.duration_mean     30    0.000056     0.000061     0.000065     0.000079
...     ...                      ...                                  ...    ...         ...          ...          ...          ...
53      log  recommendationservice-2                        log.row_count     30  196.000000   241.000000   260.550000   270.000000
40      log         paymentservice-2                      log.error_count     30    0.000000     0.000000     0.000000     0.000000
41      log         paymentservice-2                        log.row_count     30    6.000000    12.500000    19.650000    21.000000
42      log  productcatalogservice-0                      log.error_count     30    0.000000     0.000000     0.000000     0.000000
43      log  productcatalogservice-0                        log.row_count     30  946.000000  1144.500000  1228.700000  1292.000000

[50 rows x 8 columns],          file               min_ts_utc8               max_ts_utc8
0      metric 2022-03-20 14:00:00+08:00 2022-03-20 14:29:00+08:00
1       trace 2022-03-20 14:00:00+08:00 2022-03-20 14:29:00+08:00
2         log 2022-03-20 14:00:00+08:00 2022-03-20 14:29:00+08:00
3  error_logs 2022-03-20 14:00:00+08:00 2022-03-20 14:29:59+08:00, 59,                     cmdb_id  distinct_kpis
0                 adservice            341
1                adservice2            333
2                frontend-0            254
3                frontend-1            250
4                frontend-2            240
5         checkoutservice-0            220
6         checkoutservice-1            212
7         checkoutservice-2            212
8               frontend2-0            196
9   productcatalogservice-0            158
10  productcatalogservice-2            158
11  productcatalogservice-1            154
12       checkoutservice2-0            146
13            cartservice-1            122
14            cartservice-0            118
15            cartservice-2            114
16  recommendationservice-0            110
17  recommendationservice-2            110
18  recommendationservice-1            110
19        shippingservice-2            106)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Load metric.csv (reuse if already present in kernel)
metric_fp = 'metric.csv'
df_metric = pd.read_csv(metric_fp) if 'df_metric' not in globals() else df_metric

# Ensure proper dtypes
df_metric['timestamp'] = pd.to_numeric(df_metric['timestamp'], errors='coerce').astype('Int64')
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')

# 1) Compute global thresholds across the FULL metric series per (cmdb_id, kpi_name)
metric_thresholds = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    .agg(
        count = lambda x: int(x.count()),
        min = lambda x: float(x.min()) if x.count()>0 else np.nan,
        p5  = lambda x: float(x.quantile(0.05)) if x.count()>0 else np.nan,
        p50 = lambda x: float(x.quantile(0.50)) if x.count()>0 else np.nan,
        p95 = lambda x: float(x.quantile(0.95)) if x.count()>0 else np.nan,
        max = lambda x: float(x.max()) if x.count()>0 else np.nan
    )
    .reset_index()
)

# 2) Filter metric data to the incident window 2022-03-20 14:00:00 to 2022-03-20 14:29:59 (UTC+8)
start_dt = pd.Timestamp('2022-03-20 14:00:00', tz=tz).astimezone(pytz.UTC)
end_dt   = pd.Timestamp('2022-03-20 14:29:59', tz=tz).astimezone(pytz.UTC)
start_ts = int(start_dt.timestamp())
end_ts   = int(end_dt.timestamp())

metric_incident = df_metric[(df_metric['timestamp'] >= start_ts) & (df_metric['timestamp'] <= end_ts)].copy()

# 3) Detect anomaly points in the incident window using global p95/p5 thresholds
# Join thresholds onto incident points
metric_incident_with_thr = metric_incident.merge(
    metric_thresholds[['cmdb_id','kpi_name','p95','p5']],
    on=['cmdb_id','kpi_name'],
    how='left'
)

# Define anomaly flag where value >= p95 OR value <= p5 (handle missing thresholds conservatively: not anomaly)
metric_incident_with_thr['is_anomaly'] = False
mask_thr = metric_incident_with_thr['p95'].notna() & metric_incident_with_thr['p5'].notna() & metric_incident_with_thr['value'].notna()
metric_incident_with_thr.loc[mask_thr, 'is_anomaly'] = (
    (metric_incident_with_thr.loc[mask_thr, 'value'] >= metric_incident_with_thr.loc[mask_thr, 'p95']) |
    (metric_incident_with_thr.loc[mask_thr, 'value'] <= metric_incident_with_thr.loc[mask_thr, 'p5'])
)

# Aggregate anomalies per series that have >=1 anomaly in window
anomaly_groups = metric_incident_with_thr[metric_incident_with_thr['is_anomaly']].groupby(['cmdb_id','kpi_name'], dropna=False)

anomalies_df = anomaly_groups.agg(
    threshold_p95 = ('p95', 'first'),
    threshold_p5  = ('p5', 'first'),
    earliest_anomaly_ts = ('timestamp', 'min'),
    anomaly_count_in_window = ('is_anomaly', 'sum'),
    max_value_in_window = ('value', 'max'),
    min_value_in_window = ('value', 'min')
).reset_index()

# Compute severity_score per definition
def compute_severity(row):
    t95 = row['threshold_p95'] if pd.notna(row['threshold_p95']) else 0.0
    t5  = row['threshold_p5'] if pd.notna(row['threshold_p5']) else 0.0
    max_v = row['max_value_in_window'] if pd.notna(row['max_value_in_window']) else 0.0
    min_v = row['min_value_in_window'] if pd.notna(row['min_value_in_window']) else 0.0
    term1 = (max_v / t95) if (t95>0) else 0.0
    term2 = (t5 / min_v) if (min_v>0) else 0.0
    return max(term1, term2)

if not anomalies_df.empty:
    anomalies_df['severity_score'] = anomalies_df.apply(compute_severity, axis=1)
    # Convert earliest_anomaly_ts to UTC+8 human-readable
    anomalies_df['earliest_anomaly_timestamp_utc8'] = anomalies_df['earliest_anomaly_ts'].apply(
        lambda x: pd.to_datetime(int(x), unit='s', utc=True).tz_convert(tz) if pd.notna(x) else pd.NaT
    )
else:
    anomalies_df['severity_score'] = pd.Series(dtype=float)
    anomalies_df['earliest_anomaly_timestamp_utc8'] = pd.Series(dtype='datetime64[ns, Asia/Shanghai]')

# 4) Return top 20 series with anomalies sorted by severity_score desc.
anomalies_top20 = (
    anomalies_df
    .sort_values('severity_score', ascending=False)
    .loc[:, ['cmdb_id','kpi_name','threshold_p95','threshold_p5','earliest_anomaly_timestamp_utc8',
             'anomaly_count_in_window','max_value_in_window','min_value_in_window','severity_score']]
    .head(20)
    .reset_index(drop=True)
)

# Totals
total_series_scanned = int(metric_thresholds.shape[0])  # number of distinct (cmdb_id,kpi_name) in full metric series
total_series_with_anomalies = int(anomalies_df.shape[0])

# Display compact aggregates only
anomalies_top20, total_series_scanned, total_series_with_anomalies

```
Out[2]:
```
# Build a concise plain-English summary string using existing kernel variables
# Assumes anomalies_top20, total_series_scanned, total_series_with_anomalies exist from prior run.

top_n = 5
top = anomalies_top20.head(top_n) if 'anomalies_top20' in globals() else None
scanned = total_series_scanned if 'total_series_scanned' in globals() else None
with_anoms = total_series_with_anomalies if 'total_series_with_anomalies' in globals() else None

summary_lines = []
summary_lines.append(f"Incident window: 2022-03-20 14:00:00 to 2022-03-20 14:29:59 (UTC+8)")
summary_lines.append(f"Total metric series scanned: {scanned}")
summary_lines.append(f"Number of series with ≥1 anomaly in window: {with_anoms}")
summary_lines.append("")
summary_lines.append(f"Top {top_n} anomalous series by severity (fields: cmdb_id | kpi_name | earliest_anomaly_utc8 | anomalies_in_window | min | max | severity):")

if top is None or top.shape[0]==0:
    summary_lines.append("  (no anomalies detected)")
else:
    for i, row in top.iterrows():
        cmdb = row['cmdb_id']
        kpi = row['kpi_name']
        ts = row['earliest_anomaly_timestamp_utc8']
        cnt = int(row['anomaly_count_in_window'])
        vmin = row['min_value_in_window']
        vmax = row['max_value_in_window']
        sev = float(row['severity_score'])
        summary_lines.append(
            f"  {i+1}) {cmdb} | {kpi} | {ts} | count={cnt} | min={vmin:.6g} | max={vmax:.6g} | severity={sev:.4f}"
        )

summary = "\n".join(summary_lines)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(               cmdb_id                                           kpi_name  threshold_p95  threshold_p5 earliest_anomaly_timestamp_utc8  anomaly_count_in_window  max_value_in_window  min_value_in_window  severity_score
0    currencyservice-1  container.node-5.container_cpu_cfs_throttled_s...       0.001056      0.000000       2022-03-20 14:00:00+08:00                       30         4.710503e-02             0.000000       44.616655
1    checkoutservice-0      container.node-5.container_memory_mapped_file   73728.000000  11673.600000       2022-03-20 14:14:00+08:00                        9         3.186688e+06             0.000000       43.222222
2   currencyservice2-0  container.node-6.container_cpu_cfs_throttled_s...       0.273663      0.010463       2022-03-20 14:10:00+08:00                        4         4.303439e-01             0.000279       37.507886
3     paymentservice-1  container.node-5.container_network_receive_MB....       0.025181      0.018604       2022-03-20 14:03:00+08:00                        4         5.498619e-01             0.017534       21.836705
4    paymentservice2-0  container.node-6.container_network_receive_MB....       0.025657      0.015073       2022-03-20 14:00:00+08:00                        5         5.565750e-01             0.014022       21.693163
5     paymentservice-0  container.node-5.container_network_receive_MB....       0.025848      0.019101       2022-03-20 14:02:00+08:00                        4         5.565271e-01             0.018534       21.530359
6    checkoutservice-1      container.node-5.container_fs_writes./dev/vda       1.650000      0.000000       2022-03-20 14:00:00+08:00                       30         3.300000e+01             0.000000       20.000000
7    checkoutservice-0      container.node-5.container_fs_writes./dev/vda       1.650000      0.000000       2022-03-20 14:00:00+08:00                       30         3.200000e+01             0.000000       19.393939
8   checkoutservice2-0      container.node-5.container_fs_writes./dev/vda       1.650000      0.000000       2022-03-20 14:00:00+08:00                       30         3.200000e+01             0.000000       19.393939
9       emailservice-0  container.node-5.container_network_receive_MB....       0.030718      0.022016       2022-03-20 14:02:00+08:00                        4         5.558000e-01             0.017328       18.093825
10   checkoutservice-2      container.node-5.container_fs_writes./dev/vda       2.200000      0.000000       2022-03-20 14:00:00+08:00                       30         3.600000e+01             0.000000       16.363636
11   checkoutservice-1   container.node-5.container_fs_writes_MB./dev/vda       0.015039      0.000000       2022-03-20 14:00:00+08:00                       30         2.265625e-01             0.000000       15.064935
12   checkoutservice-0   container.node-5.container_fs_writes_MB./dev/vda       0.015039      0.000000       2022-03-20 14:00:00+08:00                       30         2.265625e-01             0.000000       15.064935
13  checkoutservice2-0   container.node-5.container_fs_writes_MB./dev/vda       0.015039      0.000000       2022-03-20 14:00:00+08:00                       30         2.226562e-01             0.000000       14.805195
14  shippingservice2-0  container.node-5.container_network_receive_MB....       0.037989      0.025584       2022-03-20 14:00:00+08:00                        5         5.619049e-01             0.024023       14.791431
15   checkoutservice-2   container.node-5.container_fs_writes_MB./dev/vda       0.015039      0.000000       2022-03-20 14:00:00+08:00                       30         2.187500e-01             0.000000       14.545455
16   shippingservice-1  container.node-5.container_network_receive_MB....       0.046082      0.032285       2022-03-20 14:02:00+08:00                        4         5.725284e-01             0.031007       12.423994
17              node-4                                      system.io.w_s       0.775000      0.000000       2022-03-20 14:01:00+08:00                       29         9.500000e+00             0.000000       12.258065
18   shippingservice-0  container.node-5.container_network_receive_MB....       0.046404      0.035052       2022-03-20 14:02:00+08:00                        4         5.687885e-01             0.030012       12.257416
19   shippingservice-2  container.node-5.container_network_receive_MB....       0.046931      0.032033       2022-03-20 14:02:00+08:00                        4         5.671363e-01             0.031284       12.084359, 4818, 4818)```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing kernel variables if present, else load/recompute minimally
if 'metric_incident' not in globals() or 'metric_thresholds' not in globals():
    # Load metric.csv and compute metric_thresholds and metric_incident as in prior steps
    df_metric = pd.read_csv('metric.csv')
    df_metric['timestamp'] = pd.to_numeric(df_metric['timestamp'], errors='coerce').astype('Int64')
    df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
    metric_thresholds = (
        df_metric
        .groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
        .agg(
            count = lambda x: int(x.count()),
            min = lambda x: float(x.min()) if x.count()>0 else np.nan,
            p5  = lambda x: float(x.quantile(0.05)) if x.count()>0 else np.nan,
            p50 = lambda x: float(x.quantile(0.50)) if x.count()>0 else np.nan,
            p95 = lambda x: float(x.quantile(0.95)) if x.count()>0 else np.nan,
            max = lambda x: float(x.max()) if x.count()>0 else np.nan
        )
        .reset_index()
    )
    start_ts = int(pd.Timestamp('2022-03-20 14:00:00', tz=tz).astimezone(pytz.UTC).timestamp())
    end_ts   = int(pd.Timestamp('2022-03-20 14:29:59', tz=tz).astimezone(pytz.UTC).timestamp())
    metric_incident = df_metric[(df_metric['timestamp'] >= start_ts) & (df_metric['timestamp'] <= end_ts)].copy()

# Prepare incident points with thresholds joined
mi = metric_incident.merge(
    metric_thresholds[['cmdb_id','kpi_name','p95','p5']],
    on=['cmdb_id','kpi_name'],
    how='left'
).copy()

# Mark anomaly points: require thresholds present and value non-null
mi['is_anomaly'] = False
mask_valid = mi['p95'].notna() & mi['p5'].notna() & mi['value'].notna()
mi.loc[mask_valid, 'is_anomaly'] = (
    (mi.loc[mask_valid, 'value'] >= mi.loc[mask_valid, 'p95']) |
    (mi.loc[mask_valid, 'value'] <= mi.loc[mask_valid, 'p5'])
)

# Extract only anomaly points and sort
anom_pts = mi[mi['is_anomaly']].copy()
anom_pts.sort_values(['cmdb_id','kpi_name','timestamp'], inplace=True)

# If no anomaly points, prepare empty outputs
if anom_pts.empty:
    metric_fault_runs = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','run_start_timestamp_utc8','run_end_timestamp_utc8',
        'run_length_minutes','max_value_in_run','min_value_in_run',
        'threshold_p95','threshold_p5','run_severity'
    ])
    metric_fault_runs_top20 = metric_fault_runs.copy().head(20)
    metric_fault_runs_by_cmdb = pd.DataFrame(columns=['cmdb_id','number_of_fault_runs','total_fault_minutes','max_run_severity']).head(10)
else:
    # For each series, identify consecutive runs where timestamp gaps <= 60s
    # Compute time diff per series
    anom_pts['ts_diff'] = anom_pts.groupby(['cmdb_id','kpi_name'])['timestamp'].diff().fillna(0).astype('Int64')
    # New run where diff > 60
    anom_pts['new_run'] = (anom_pts['ts_diff'] > 60).astype(int)
    # Cumulative run id per series
    anom_pts['run_id'] = anom_pts.groupby(['cmdb_id','kpi_name'])['new_run'].cumsum()
    # Aggregate per (cmdb_id,kpi_name,run_id)
    agg = anom_pts.groupby(['cmdb_id','kpi_name','run_id'], dropna=False).agg(
        run_start_ts = ('timestamp','min'),
        run_end_ts   = ('timestamp','max'),
        run_length_minutes = ('timestamp','count'),
        max_value_in_run = ('value','max'),
        min_value_in_run = ('value','min'),
        threshold_p95 = ('p95','first'),
        threshold_p5  = ('p5','first')
    ).reset_index(drop=False)
    # Compute run_severity
    def run_sev(row):
        t95 = row['threshold_p95'] if pd.notna(row['threshold_p95']) else 0.0
        t5  = row['threshold_p5'] if pd.notna(row['threshold_p5']) else 0.0
        max_v = row['max_value_in_run'] if pd.notna(row['max_value_in_run']) else 0.0
        min_v = row['min_value_in_run'] if pd.notna(row['min_value_in_run']) else 0.0
        term1 = (max_v / t95) if (t95>0) else 0.0
        term2 = (t5 / min_v) if (min_v>0) else 0.0
        return max(term1, term2)
    agg['run_severity'] = agg.apply(run_sev, axis=1)
    # Convert start/end to UTC+8 timestamps
    agg['run_start_timestamp_utc8'] = agg['run_start_ts'].apply(lambda x: pd.to_datetime(int(x), unit='s', utc=True).tz_convert(tz))
    agg['run_end_timestamp_utc8']   = agg['run_end_ts'].apply(lambda x: pd.to_datetime(int(x), unit='s', utc=True).tz_convert(tz))
    # Final columns and store full runs table
    metric_fault_runs = agg[[
        'cmdb_id','kpi_name','run_start_timestamp_utc8','run_end_timestamp_utc8',
        'run_length_minutes','max_value_in_run','min_value_in_run',
        'threshold_p95','threshold_p5','run_severity'
    ]].copy()
    # Prepare top20 runs by severity desc
    metric_fault_runs_top20 = metric_fault_runs.sort_values('run_severity', ascending=False).head(20).reset_index(drop=True)
    # Aggregate per cmdb_id
    metric_fault_runs_by_cmdb = (
        metric_fault_runs
        .groupby('cmdb_id', dropna=False)
        .agg(
            number_of_fault_runs = ('run_length_minutes','count'),
            total_fault_minutes = ('run_length_minutes','sum'),
            max_run_severity = ('run_severity','max')
        )
        .reset_index()
        .sort_values(['number_of_fault_runs','max_run_severity'], ascending=[False, False])
        .head(10)
        .reset_index(drop=True)
    )

# Display only the two compact tables requested
metric_fault_runs_top20, metric_fault_runs_by_cmdb

```
Out[3]:
```
Summary (incident window 2022-03-20 14:00–14:29:59 UTC+8):

- Overall finding: multiple consecutive anomaly runs were detected across many metric series during the incident window. Runs are sequences of anomaly points where each consecutive timestamp gap ≤ 60s.

- Most severe runs (top 3 by run_severity):
  1) currencyservice-1 — metric "container.node-5.container_cpu_cfs_throttled_..."  
     - Run: 14:00 → 14:29 (30 minutes)  
     - max=0.047105, min=0.000000, threshold_p95=0.001056, p5=0.000000  
     - run_severity ≈ 44.62
  2) checkoutservice-0 — metric "container.node-5.container_memory_mapped_file"  
     - Run: 14:14 → 14:16 (3 minutes)  
     - max=3,186,688, min=0.000000, threshold_p95=73,728, p5=11,673.6  
     - run_severity ≈ 43.22
  3) currencyservice2-0 — metric "container.node-6.container_cpu_cfs_throttled_..."  
     - Run: 14:10 → 14:11 (2 minutes)  
     - max=0.430344, min=0.000279, threshold_p95=0.273663, p5=0.010463  
     - run_severity ≈ 37.51

- Examples of other notable runs in the top 20:
  - Several checkoutservice series show long (30-min) runs on filesystem write metrics with high severity (~15–20).
  - Multiple paymentservice and shippingservice network-receive metrics appear with high severity (~12–22) for short runs.
  - node-4 system.io.w_s shows a long run (14:01 → 14:29, 29 minutes) with run_severity ≈ 12.26.

- Per-cmdb summary (top 10 by number_of_fault_runs):
  - adservice: 449 fault runs (total_fault_minutes 9177), max_run_severity ≈ 1.31
  - adservice2: 411 runs (8993 minutes), max_run_severity ≈ 1.82
  - checkoutservice-0: 398 runs (2335 minutes), highest run_severity ≈ 43.22 (one of the most severe)
  - frontend-0 / frontend-1 / frontend2-0 / frontend-2, checkoutservice-2, checkoutservice2-0, productcatalogservice-0 also show many runs (see top-10).  
  - Note: some components (e.g., adservice) have many runs but low max severity; others (checkoutservice-0, currencyservice-1) have fewer runs but very high severity.

Interpretation pointers:
- The highest-severity runs point to CPU throttling metrics (currencyservice*), large mapped-memory values (checkoutservice-0), filesystem write spikes (checkoutservice*), and elevated network receive metrics (payment/shipping). These are the priority faults to investigate further.
- Components with very high counts of short/low-severity runs (e.g., adservice) may indicate frequent but low-impact anomalies or noisy signals; components with high run_severity require urgent attention.

If you want, I can:
- show the full top-20 runs table or per-cmdb table again,
- or filter runs by a specific service or severity threshold for focused investigation.

The original code execution output of IPython Kernel is also provided below for reference:

(               cmdb_id                                           kpi_name  run_start_timestamp_utc8    run_end_timestamp_utc8  run_length_minutes  max_value_in_run  min_value_in_run  threshold_p95  threshold_p5  run_severity
0    currencyservice-1  container.node-5.container_cpu_cfs_throttled_s... 2022-03-20 14:00:00+08:00 2022-03-20 14:29:00+08:00                  30      4.710503e-02          0.000000       0.001056      0.000000     44.616655
1    checkoutservice-0      container.node-5.container_memory_mapped_file 2022-03-20 14:14:00+08:00 2022-03-20 14:16:00+08:00                   3      3.186688e+06          0.000000   73728.000000  11673.600000     43.222222
2   currencyservice2-0  container.node-6.container_cpu_cfs_throttled_s... 2022-03-20 14:10:00+08:00 2022-03-20 14:11:00+08:00                   2      4.303439e-01          0.000279       0.273663      0.010463     37.507886
3     paymentservice-1  container.node-5.container_network_receive_MB.... 2022-03-20 14:19:00+08:00 2022-03-20 14:20:00+08:00                   2      5.498619e-01          0.017534       0.025181      0.018604     21.836705
4    paymentservice2-0  container.node-6.container_network_receive_MB.... 2022-03-20 14:26:00+08:00 2022-03-20 14:26:00+08:00                   1      5.565750e-01          0.556575       0.025657      0.015073     21.693163
5     paymentservice-0  container.node-5.container_network_receive_MB.... 2022-03-20 14:15:00+08:00 2022-03-20 14:16:00+08:00                   2      5.565271e-01          0.027167       0.025848      0.019101     21.530359
6    checkoutservice-1      container.node-5.container_fs_writes./dev/vda 2022-03-20 14:00:00+08:00 2022-03-20 14:29:00+08:00                  30      3.300000e+01          0.000000       1.650000      0.000000     20.000000
7   checkoutservice2-0      container.node-5.container_fs_writes./dev/vda 2022-03-20 14:00:00+08:00 2022-03-20 14:29:00+08:00                  30      3.200000e+01          0.000000       1.650000      0.000000     19.393939
8    checkoutservice-0      container.node-5.container_fs_writes./dev/vda 2022-03-20 14:00:00+08:00 2022-03-20 14:29:00+08:00                  30      3.200000e+01          0.000000       1.650000      0.000000     19.393939
9       emailservice-0  container.node-5.container_network_receive_MB.... 2022-03-20 14:19:00+08:00 2022-03-20 14:19:00+08:00                   1      5.558000e-01          0.555800       0.030718      0.022016     18.093825
10   checkoutservice-2      container.node-5.container_fs_writes./dev/vda 2022-03-20 14:00:00+08:00 2022-03-20 14:29:00+08:00                  30      3.600000e+01          0.000000       2.200000      0.000000     16.363636
11   checkoutservice-0   container.node-5.container_fs_writes_MB./dev/vda 2022-03-20 14:00:00+08:00 2022-03-20 14:29:00+08:00                  30      2.265625e-01          0.000000       0.015039      0.000000     15.064935
12   checkoutservice-1   container.node-5.container_fs_writes_MB./dev/vda 2022-03-20 14:00:00+08:00 2022-03-20 14:29:00+08:00                  30      2.265625e-01          0.000000       0.015039      0.000000     15.064935
13  checkoutservice2-0   container.node-5.container_fs_writes_MB./dev/vda 2022-03-20 14:00:00+08:00 2022-03-20 14:29:00+08:00                  30      2.226562e-01          0.000000       0.015039      0.000000     14.805195
14  shippingservice2-0  container.node-5.container_network_receive_MB.... 2022-03-20 14:22:00+08:00 2022-03-20 14:22:00+08:00                   1      5.619049e-01          0.561905       0.037989      0.025584     14.791431
15   checkoutservice-2   container.node-5.container_fs_writes_MB./dev/vda 2022-03-20 14:00:00+08:00 2022-03-20 14:29:00+08:00                  30      2.187500e-01          0.000000       0.015039      0.000000     14.545455
16   shippingservice-1  container.node-5.container_network_receive_MB.... 2022-03-20 14:11:00+08:00 2022-03-20 14:12:00+08:00                   2      5.725284e-01          0.031400       0.046082      0.032285     12.423994
17              node-4                                      system.io.w_s 2022-03-20 14:01:00+08:00 2022-03-20 14:29:00+08:00                  29      9.500000e+00          0.000000       0.775000      0.000000     12.258065
18   shippingservice-0  container.node-5.container_network_receive_MB.... 2022-03-20 14:29:00+08:00 2022-03-20 14:29:00+08:00                   1      5.687885e-01          0.568789       0.046404      0.035052     12.257416
19   shippingservice-2  container.node-5.container_network_receive_MB.... 2022-03-20 14:13:00+08:00 2022-03-20 14:13:00+08:00                   1      5.671363e-01          0.567136       0.046931      0.032033     12.084359,                    cmdb_id  number_of_fault_runs  total_fault_minutes  max_run_severity
0                adservice                   449                 9177          1.310252
1               adservice2                   411                 8993          1.818182
2        checkoutservice-0                   398                 2335         43.222222
3               frontend-0                   354                 2920          2.004796
4               frontend-1                   342                 2808          2.327255
5              frontend2-0                   335                 3035          2.942967
6        checkoutservice-2                   334                 1955         16.363636
7               frontend-2                   324                 2520          1.394762
8       checkoutservice2-0                   290                 1716         19.393939
9  productcatalogservice-0                   253                 2341          2.713633)```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Load trace.csv (reuse if available in kernel)
if 'df_trace' not in globals():
    df_trace = pd.read_csv('trace.csv')

# Ensure dtypes
df_trace['timestamp'] = pd.to_numeric(df_trace['timestamp'], errors='coerce').astype('Int64')
df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')

# 1) Compute global thresholds (P95 and P5) for each (cmdb_id, trace_name)
trace_thresholds = (
    df_trace
    .groupby(['cmdb_id', 'trace_name'], dropna=False)['value']
    .agg(
        count = lambda x: int(x.count()),
        p5 = lambda x: float(x.quantile(0.05)) if x.count()>0 else np.nan,
        p95 = lambda x: float(x.quantile(0.95)) if x.count()>0 else np.nan,
        min = lambda x: float(x.min()) if x.count()>0 else np.nan,
        max = lambda x: float(x.max()) if x.count()>0 else np.nan
    )
    .reset_index()
)

# 2) Filter trace.csv to incident window (UTC+8) -> trace_incident
start_ts = int(pd.Timestamp('2022-03-20 14:00:00', tz=tz).astimezone(pytz.UTC).timestamp())
end_ts   = int(pd.Timestamp('2022-03-20 14:29:59', tz=tz).astimezone(pytz.UTC).timestamp())
trace_incident = df_trace[(df_trace['timestamp'] >= start_ts) & (df_trace['timestamp'] <= end_ts)].copy()

# 3) Mark anomaly points using thresholds (value >= p95 OR value <= p5)
ti = trace_incident.merge(
    trace_thresholds[['cmdb_id','trace_name','p95','p5']],
    on=['cmdb_id','trace_name'],
    how='left'
).copy()

# Only mark anomalies where thresholds exist and value present
ti['is_anomaly'] = False
mask_valid = ti['p95'].notna() & ti['p5'].notna() & ti['value'].notna()
ti.loc[mask_valid, 'is_anomaly'] = (
    (ti.loc[mask_valid, 'value'] >= ti.loc[mask_valid, 'p95']) |
    (ti.loc[mask_valid, 'value'] <= ti.loc[mask_valid, 'p5'])
)

# Aggregate per series that have >=1 anomaly in the window
anom_grp = ti[ti['is_anomaly']].groupby(['cmdb_id','trace_name'], dropna=False)

anomalies = anom_grp.agg(
    threshold_p95 = ('p95','first'),
    threshold_p5  = ('p5','first'),
    earliest_anomaly_ts = ('timestamp','min'),
    anomaly_count_in_window = ('is_anomaly','sum'),
    max_value_in_window = ('value','max'),
    min_value_in_window = ('value','min')
).reset_index()

# Compute severity_score
def compute_severity(row):
    t95 = row['threshold_p95'] if pd.notna(row['threshold_p95']) else 0.0
    t5  = row['threshold_p5'] if pd.notna(row['threshold_p5']) else 0.0
    max_v = row['max_value_in_window'] if pd.notna(row['max_value_in_window']) else 0.0
    min_v = row['min_value_in_window'] if pd.notna(row['min_value_in_window']) else 0.0
    term1 = (max_v / t95) if (t95>0) else 0.0
    term2 = (t5 / min_v) if (min_v>0) else 0.0
    return max(term1, term2)

if not anomalies.empty:
    anomalies['severity_score'] = anomalies.apply(compute_severity, axis=1)
    anomalies['earliest_anomaly_timestamp_utc8'] = anomalies['earliest_anomaly_ts'].apply(
        lambda x: pd.to_datetime(int(x), unit='s', utc=True).tz_convert(tz)
    )
else:
    anomalies['severity_score'] = pd.Series(dtype=float)
    anomalies['earliest_anomaly_timestamp_utc8'] = pd.Series(dtype='datetime64[ns, Asia/Shanghai]')

# Prepare compact top-20 table sorted by severity desc
trace_anomalies_top20 = (
    anomalies
    .sort_values('severity_score', ascending=False)
    .loc[:, ['cmdb_id','trace_name','threshold_p95','threshold_p5','earliest_anomaly_timestamp_utc8',
             'anomaly_count_in_window','max_value_in_window','min_value_in_window','severity_score']]
    .head(20)
    .reset_index(drop=True)
)

# Scalars
total_trace_series_scanned = int(trace_thresholds.shape[0])
number_of_trace_series_with_anomalies_in_window = int(anomalies.shape[0])

# Display compact results (store trace_thresholds and trace_incident in kernel as requested)
trace_thresholds, trace_incident, trace_anomalies_top20, total_trace_series_scanned, number_of_trace_series_with_anomalies_in_window

```
Out[4]:
```
# Summarize trace anomaly results as a single string (reuse kernel variables)
topn = 3
top = trace_anomalies_top20.head(topn) if 'trace_anomalies_top20' in globals() else None
total_scanned = total_trace_series_scanned if 'total_trace_series_scanned' in globals() else None
num_with_anoms = number_of_trace_series_with_anomalies_in_window if 'number_of_trace_series_with_anomalies_in_window' in globals() else None

summary_lines = []
summary_lines.append("Trace anomaly summary (incident window 2022-03-20 14:00:00 to 14:29:59 UTC+8):")
summary_lines.append(f"- Total trace series scanned: {total_scanned}")
summary_lines.append(f"- Number of trace series with ≥1 anomaly in window: {num_with_anoms}")
summary_lines.append("")
if top is None or top.shape[0]==0:
    summary_lines.append("- No anomalous trace series detected in the window.")
else:
    summary_lines.append(f"- Top {topn} anomalous trace series by severity:")
    for i, row in top.iterrows():
        cmdb = row['cmdb_id']
        name = row['trace_name']
        ts = row['earliest_anomaly_timestamp_utc8']
        cnt = int(row['anomaly_count_in_window'])
        vmax = row['max_value_in_window']
        vmin = row['min_value_in_window']
        p95 = row['threshold_p95']
        p5  = row['threshold_p5']
        sev = float(row['severity_score'])
        summary_lines.append(
            f"  {i+1}) {cmdb} | {name} | earliest={ts} | count={cnt} | min={vmin:.6g} | max={vmax:.6g} | p95={p95:.6g} | p5={p5:.6g} | severity={sev:.4f}"
        )

summary_lines.append("")
summary_lines.append("Key observation: High-severity anomalies concentrate on checkout/cart related traces (duration_mean and duration_p95), with the top series showing sustained spikes beginning at 14:00. Investigate checkout→cart and related service latency paths and host-level resource issues for root cause.")
summary = "\n".join(summary_lines)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(                 cmdb_id                               trace_name  count         p5        p95        min        max
0            adservice-0      trace.from_frontend-0.duration_mean     30   0.000012   0.000014   0.000011   0.000021
1            adservice-0       trace.from_frontend-0.duration_p95     30   0.000013   0.000021   0.000012   0.000028
2            adservice-0         trace.from_frontend-0.error_rate     30   0.000000   0.000000   0.000000   0.000000
3            adservice-0          trace.from_frontend-0.row_count     30  18.450000  26.550000  15.000000  29.000000
4            adservice-0      trace.from_frontend-1.duration_mean     30   0.000011   0.000014   0.000011   0.000024
...                  ...                                      ...    ...        ...        ...        ...        ...
1227  shippingservice2-0  trace.from_checkoutservice2-0.row_count     26   2.000000  10.000000   2.000000  10.000000
1228  shippingservice2-0     trace.from_frontend2-0.duration_mean     30   0.000059   0.000082   0.000058   0.000102
1229  shippingservice2-0      trace.from_frontend2-0.duration_p95     30   0.000062   0.000191   0.000060   0.000272
1230  shippingservice2-0        trace.from_frontend2-0.error_rate     30   0.000000   0.000000   0.000000   0.000000
1231  shippingservice2-0         trace.from_frontend2-0.row_count     30   5.000000  20.550000   3.000000  22.000000

[1232 rows x 7 columns],         timestamp             cmdb_id                            trace_name      value
0      1647756000         adservice-0   trace.from_frontend-0.duration_mean   0.000021
1      1647756000         adservice-0    trace.from_frontend-0.duration_p95   0.000028
2      1647756000         adservice-0      trace.from_frontend-0.error_rate   0.000000
3      1647756000         adservice-0       trace.from_frontend-0.row_count  20.000000
4      1647756000         adservice-0   trace.from_frontend-1.duration_mean   0.000012
...           ...                 ...                                   ...        ...
36531  1647757740   shippingservice-2       trace.from_frontend-2.row_count   8.000000
36532  1647757740  shippingservice2-0  trace.from_frontend2-0.duration_mean   0.000059
36533  1647757740  shippingservice2-0   trace.from_frontend2-0.duration_p95   0.000062
36534  1647757740  shippingservice2-0     trace.from_frontend2-0.error_rate   0.000000
36535  1647757740  shippingservice2-0      trace.from_frontend2-0.row_count   6.000000

[36536 rows x 4 columns],                cmdb_id                                   trace_name  threshold_p95  threshold_p5 earliest_anomaly_timestamp_utc8  anomaly_count_in_window  max_value_in_window  min_value_in_window  severity_score
0        cartservice-0   trace.from_checkoutservice-1.duration_mean       0.000183      0.000000       2022-03-20 14:00:00+08:00                       30             0.009000             0.000000       49.140049
1    checkoutservice-1         trace.to_cartservice-0.duration_mean       0.000183      0.000000       2022-03-20 14:00:00+08:00                       30             0.009000             0.000000       49.140049
2    checkoutservice-1          trace.to_cartservice-0.duration_p95       0.000495      0.000000       2022-03-20 14:00:00+08:00                       30             0.017100             0.000000       34.545455
3        cartservice-0    trace.from_checkoutservice-1.duration_p95       0.000495      0.000000       2022-03-20 14:00:00+08:00                       30             0.017100             0.000000       34.545455
4    checkoutservice-2         trace.to_cartservice-2.duration_mean       0.000137      0.000000       2022-03-20 14:00:00+08:00                       30             0.002333             0.000000       16.967273
5        cartservice-2   trace.from_checkoutservice-2.duration_mean       0.000137      0.000000       2022-03-20 14:00:00+08:00                       30             0.002333             0.000000       16.967273
6        cartservice-0          trace.from_frontend-2.duration_mean       0.000067      0.000000       2022-03-20 14:01:00+08:00                       19             0.001059             0.000000       15.712166
7           frontend-2         trace.to_cartservice-0.duration_mean       0.000067      0.000000       2022-03-20 14:01:00+08:00                       19             0.001059             0.000000       15.712166
8        cartservice-2    trace.from_checkoutservice-2.duration_p95       0.000467      0.000000       2022-03-20 14:00:00+08:00                       30             0.006300             0.000000       13.475936
9    checkoutservice-2          trace.to_cartservice-2.duration_p95       0.000467      0.000000       2022-03-20 14:00:00+08:00                       30             0.006300             0.000000       13.475936
10  checkoutservice2-0     trace.to_currencyservice2-0.duration_p95       0.010194      0.001193       2022-03-20 14:10:00+08:00                        4             0.012311             0.000120        9.941667
11  currencyservice2-0   trace.from_checkoutservice2-0.duration_p95       0.010194      0.001193       2022-03-20 14:10:00+08:00                        4             0.012311             0.000120        9.941667
12   checkoutservice-0                 trace.from_root.duration_p95       0.013636      0.002886       2022-03-20 14:14:00+08:00                        4             0.086927             0.002666        6.374606
13  checkoutservice2-0        trace.to_cartservice2-0.duration_mean       0.000458      0.000000       2022-03-20 14:01:00+08:00                       16             0.002500             0.000000        5.455537
14      cartservice2-0  trace.from_checkoutservice2-0.duration_mean       0.000458      0.000000       2022-03-20 14:01:00+08:00                       16             0.002500             0.000000        5.455537
15  checkoutservice2-0                      trace.self.duration_p95       0.011298      0.006932       2022-03-20 14:08:00+08:00                        4             0.055282             0.006337        4.892862
16   shippingservice-0    trace.from_checkoutservice-1.duration_p95       0.000073      0.000057       2022-03-20 14:08:00+08:00                        4             0.000356             0.000056        4.886754
17   checkoutservice-1      trace.to_shippingservice-0.duration_p95       0.000073      0.000057       2022-03-20 14:08:00+08:00                        4             0.000356             0.000056        4.886754
18   currencyservice-0    trace.from_checkoutservice-0.duration_p95       0.000113      0.000096       2022-03-20 14:24:00+08:00                        4             0.000526             0.000093        4.669330
19   checkoutservice-0      trace.to_currencyservice-0.duration_p95       0.000113      0.000096       2022-03-20 14:24:00+08:00                        4             0.000526             0.000093        4.669330, 1232, 1232)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse df_log and df_error if present, else load from CSV
if 'df_log' not in globals():
    df_log = pd.read_csv('log.csv')
if 'df_error' not in globals():
    df_error = pd.read_csv('error_logs.csv')

# Ensure correct dtypes
df_log['timestamp'] = pd.to_numeric(df_log['timestamp'], errors='coerce').astype('Int64')
df_log['value'] = pd.to_numeric(df_log['value'], errors='coerce')
df_error['timestamp'] = pd.to_numeric(df_error['timestamp'], errors='coerce').astype('Int64')

# Incident window timestamps (UTC+8 -> convert to UTC seconds)
start_ts = int(pd.Timestamp('2022-03-20 14:00:00', tz=tz).astimezone(pytz.UTC).timestamp())
end_ts   = int(pd.Timestamp('2022-03-20 14:29:59', tz=tz).astimezone(pytz.UTC).timestamp())

# 1) Log thresholds (global, full series) and incident filtering
log_thresholds = (
    df_log
    .groupby(['cmdb_id','log_name'], dropna=False)['value']
    .agg(
        count = lambda x: int(x.count()),
        p5 = lambda x: float(x.quantile(0.05)) if x.count()>0 else np.nan,
        p95 = lambda x: float(x.quantile(0.95)) if x.count()>0 else np.nan,
        min = lambda x: float(x.min()) if x.count()>0 else np.nan,
        max = lambda x: float(x.max()) if x.count()>0 else np.nan
    )
    .reset_index()
)

log_incident = df_log[(df_log['timestamp'] >= start_ts) & (df_log['timestamp'] <= end_ts)].copy()

# Join thresholds to incident points
li = log_incident.merge(
    log_thresholds[['cmdb_id','log_name','p95','p5']],
    on=['cmdb_id','log_name'],
    how='left'
)

# Mark anomalies where thresholds exist and value is not null
li['is_anomaly'] = False
mask_valid = li['p95'].notna() & li['p5'].notna() & li['value'].notna()
li.loc[mask_valid, 'is_anomaly'] = (
    (li.loc[mask_valid, 'value'] >= li.loc[mask_valid, 'p95']) |
    (li.loc[mask_valid, 'value'] <= li.loc[mask_valid, 'p5'])
)

# Aggregate anomalies per (cmdb_id, log_name)
log_anom_grp = li[li['is_anomaly']].groupby(['cmdb_id','log_name'], dropna=False)
if not log_anom_grp.ngroups:
    log_anomalies_top20 = pd.DataFrame(columns=[
        'cmdb_id','log_name','threshold_p95','threshold_p5','earliest_anomaly_timestamp_utc8',
        'anomaly_count_in_window','max_value_in_window','min_value_in_window','severity_score'
    ])
else:
    log_anomalies = log_anom_grp.agg(
        threshold_p95 = ('p95','first'),
        threshold_p5  = ('p5','first'),
        earliest_anomaly_ts = ('timestamp','min'),
        anomaly_count_in_window = ('is_anomaly','sum'),
        max_value_in_window = ('value','max'),
        min_value_in_window = ('value','min')
    ).reset_index()

    # severity calculation
    def compute_sev(row):
        t95 = row['threshold_p95'] if pd.notna(row['threshold_p95']) else 0.0
        t5  = row['threshold_p5'] if pd.notna(row['threshold_p5']) else 0.0
        max_v = row['max_value_in_window'] if pd.notna(row['max_value_in_window']) else 0.0
        min_v = row['min_value_in_window'] if pd.notna(row['min_value_in_window']) else 0.0
        term1 = (max_v / t95) if (t95>0) else 0.0
        term2 = (t5 / min_v) if (min_v>0) else 0.0
        return max(term1, term2)

    log_anomalies['severity_score'] = log_anomalies.apply(compute_sev, axis=1)
    log_anomalies['earliest_anomaly_timestamp_utc8'] = log_anomalies['earliest_anomaly_ts'].apply(
        lambda x: pd.to_datetime(int(x), unit='s', utc=True).tz_convert(tz)
    )

    log_anomalies_top20 = (
        log_anomalies
        .sort_values('severity_score', ascending=False)
        .loc[:, ['cmdb_id','log_name','threshold_p95','threshold_p5','earliest_anomaly_timestamp_utc8',
                 'anomaly_count_in_window','max_value_in_window','min_value_in_window','severity_score']]
        .head(20)
        .reset_index(drop=True)
    )

# 2) error_logs: filter to incident window and aggregate per cmdb_id
error_incident = df_error[(df_error['timestamp'] >= start_ts) & (df_error['timestamp'] <= end_ts)].copy()

if error_incident.empty:
    error_aggregates_top20 = pd.DataFrame(columns=['cmdb_id','error_count_in_window','earliest_error_timestamp_utc8','sample_messages'])
else:
    err_agg = (
        error_incident
        .groupby('cmdb_id', dropna=False)
        .agg(
            error_count_in_window = ('timestamp','count'),
            earliest_error_ts = ('timestamp','min')
        )
        .reset_index()
    )
    err_agg['earliest_error_timestamp_utc8'] = err_agg['earliest_error_ts'].apply(
        lambda x: pd.to_datetime(int(x), unit='s', utc=True).tz_convert(tz)
    )
    # Prepare sample messages for top 20; for top 5 include up to 5 distinct messages each
    err_agg = err_agg.sort_values('error_count_in_window', ascending=False).reset_index(drop=True)
    top20_err = err_agg.head(20).copy()

    # Build sample messages dict for top 5 cmdb_ids
    top5_ids = top20_err['cmdb_id'].head(5).tolist()
    samples = {}
    for cid in top5_ids:
        msgs = error_incident[error_incident['cmdb_id']==cid]['message'].dropna().astype(str).unique()[:5]
        # join messages into a single string separated by " || "
        samples[cid] = " || ".join(msgs.tolist())

    # Add sample_messages column to top20_err: populate for top5, else empty string
    top20_err['sample_messages'] = top20_err['cmdb_id'].apply(lambda x: samples.get(x, ''))
    error_aggregates_top20 = top20_err.loc[:, ['cmdb_id','error_count_in_window','earliest_error_timestamp_utc8','sample_messages']].reset_index(drop=True)

# Store full intermediate tables in kernel variables as requested:
# log_thresholds, log_incident, error_incident are stored above

# Display only the two compact tables requested
log_anomalies_top20, error_aggregates_top20

```
Out[5]:
```
# Build a concise plain-English summary from existing kernel results (log_anomalies_top20, error_aggregates_top20).
# Display the summary string only.
summary_lines = []
summary_lines.append("Log & error-log corroboration (incident window 2022-03-20 14:00:00–14:29:59 UTC+8):")
summary_lines.append("")
# Top log anomalies (brief)
if 'log_anomalies_top20' in globals() and not log_anomalies_top20.empty:
    summary_lines.append("1) Log anomalies (top examples):")
    for i, row in log_anomalies_top20.head(5).iterrows():
        summary_lines.append(
            f"   - {row['cmdb_id']} | {row['log_name']} | earliest={row['earliest_anomaly_timestamp_utc8']} "
            f"| count={int(row['anomaly_count_in_window'])} | min={row['min_value_in_window']:.6g} | max={row['max_value_in_window']:.6g} "
            f"| severity={row['severity_score']:.3f}"
        )
    summary_lines.append("   (Top log anomalies involve row_count and error_count spikes in checkout, payment, email, frontend, currency services.)")
else:
    summary_lines.append("1) No log anomalies detected in the window.")

summary_lines.append("")
# Error log aggregates
if 'error_aggregates_top20' in globals() and not error_aggregates_top20.empty:
    summary_lines.append("2) Error-log counts (top cmdb_ids):")
    for i, row in error_aggregates_top20.head(5).iterrows():
        summary_lines.append(
            f"   - {row['cmdb_id']} | errors={int(row['error_count_in_window'])} | earliest={row['earliest_error_timestamp_utc8']}"
        )
    summary_lines.append("   (Front-end components show the highest error volume early in the window — e.g., frontend-0/1/2.)")
else:
    summary_lines.append("2) No error_log entries found in the window.")

summary_lines.append("")
# Corroboration and interpretation
summary_lines.append("3) Interpretation / correlation:")
summary_lines.append("   - Log anomalies show increased request/row counts and elevated error_count across checkout, payment, email and frontend services.")
summary_lines.append("   - Error logs show large numbers of frontend errors starting at ~14:00, matching the start of anomalous traces/metrics.")
summary_lines.append("   - Together these corroborate performance/failure impacts in frontend→checkout/cart request paths and related services (checkout, cart, payment).")
summary_lines.append("   - Recommended next steps: inspect frontend error messages, investigate checkout→cart latency and backend I/O/network (filesystem writes and network receive spikes seen in metrics), and check nodes hosting the most severe metric runs (node-5/node-6).")

summary = "\n".join(summary_lines)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(                    cmdb_id         log_name  threshold_p95  threshold_p5 earliest_anomaly_timestamp_utc8  anomaly_count_in_window  max_value_in_window  min_value_in_window  severity_score
0         checkoutservice-2    log.row_count         105.60         35.35       2022-03-20 14:08:00+08:00                        4                113.0                 23.0        1.536957
1          paymentservice-1    log.row_count          18.00          9.00       2022-03-20 14:02:00+08:00                       13                 21.0                  6.0        1.500000
2            emailservice-0    log.row_count          16.55          6.90       2022-03-20 14:04:00+08:00                        4                 19.0                  5.0        1.380000
3         checkoutservice-0    log.row_count         101.00         48.00       2022-03-20 14:07:00+08:00                        7                102.0                 38.0        1.263158
4                frontend-0  log.error_count          26.55         18.45       2022-03-20 14:03:00+08:00                        4                 29.0                 15.0        1.230000
5         checkoutservice-1    log.row_count          94.65         40.30       2022-03-20 14:04:00+08:00                        4                102.0                 34.0        1.185294
6                frontend-1  log.error_count          21.55         13.90       2022-03-20 14:02:00+08:00                        4                 23.0                 12.0        1.158333
7            emailservice-2    log.row_count          17.00          8.00       2022-03-20 14:06:00+08:00                        7                 17.0                  7.0        1.142857
8         currencyservice-1    log.row_count         999.05        710.80       2022-03-20 14:02:00+08:00                        4               1041.0                628.0        1.131847
9         currencyservice-2    log.row_count         988.20        705.20       2022-03-20 14:02:00+08:00                        4               1051.0                642.0        1.098442
10               frontend-0    log.row_count        1962.35       1599.75       2022-03-20 14:02:00+08:00                        4               2093.0               1462.0        1.094220
11               frontend-2  log.error_count          25.10         17.45       2022-03-20 14:00:00+08:00                        4                 27.0                 16.0        1.090625
12        currencyservice-0    log.row_count         989.30        707.05       2022-03-20 14:02:00+08:00                        4               1043.0                650.0        1.087769
13         paymentservice-0    log.row_count          17.55          9.00       2022-03-20 14:03:00+08:00                        7                 19.0                  9.0        1.082621
14             redis-cart-0    log.row_count           6.50          5.00       2022-03-20 14:04:00+08:00                        6                  7.0                  5.0        1.076923
15         paymentservice-2    log.row_count          19.65          6.00       2022-03-20 14:01:00+08:00                        5                 21.0                  6.0        1.068702
16  recommendationservice-1    log.row_count         260.30        204.70       2022-03-20 14:03:00+08:00                        4                270.0                194.0        1.055155
17        shippingservice-2    log.row_count         138.65         78.45       2022-03-20 14:04:00+08:00                        4                146.0                 75.0        1.053011
18  productcatalogservice-0    log.row_count        1228.70        964.25       2022-03-20 14:02:00+08:00                        4               1292.0                946.0        1.051518
19  recommendationservice-2    log.row_count         260.55        205.45       2022-03-20 14:03:00+08:00                        4                270.0                196.0        1.048214,       cmdb_id  error_count_in_window earliest_error_timestamp_utc8                                    sample_messages
0  frontend-0                    686     2022-03-20 14:00:03+08:00  severity: warning, message: failed to retrieve...
1  frontend-2                    669     2022-03-20 14:00:00+08:00  severity: warning, message: failed to retrieve...
2  frontend-1                    529     2022-03-20 14:00:03+08:00  severity: warning, message: failed to retrieve...)```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse kernel variables safely
mfr = metric_fault_runs.copy() if 'metric_fault_runs' in globals() else pd.DataFrame()
tat20 = trace_anomalies_top20.copy() if 'trace_anomalies_top20' in globals() else pd.DataFrame()
err_inc = error_incident.copy() if 'error_incident' in globals() else pd.DataFrame()

# 1) Candidate A: Node/pod I/O/network incident
if not mfr.empty:
    kpi_mask = (
        mfr['kpi_name'].str.contains('fs_writes', na=False) |
        mfr['kpi_name'].str.contains('/dev/vda', na=False) |
        mfr['kpi_name'].str.contains('container_fs_writes', na=False) |
        mfr['kpi_name'].str.contains('container_network_receive', na=False)
    )
    cmdb_mask = (
        mfr['cmdb_id'].str.contains('checkoutservice', na=False) |
        mfr['cmdb_id'].str.contains('shippingservice', na=False) |
        mfr['cmdb_id'].str.contains('paymentservice', na=False) |
        mfr['cmdb_id'].str.startswith('node-', na=False)
    )
    candidateA_main = mfr[kpi_mask & cmdb_mask].copy()
    candidateA_main = candidateA_main.loc[:, ['cmdb_id','kpi_name','run_start_timestamp_utc8','run_length_minutes','max_value_in_run','threshold_p95','run_severity']]
    candidateA_main = candidateA_main.sort_values('run_severity', ascending=False).head(20).reset_index(drop=True)
    node_runs = mfr[mfr['cmdb_id'].isin(['node-4','node-5'])].copy()
    node_runs = node_runs.loc[:, ['cmdb_id','kpi_name','run_start_timestamp_utc8','run_length_minutes','max_value_in_run','threshold_p95','run_severity']]
    node_runs = node_runs.sort_values('run_severity', ascending=False).head(5).reset_index(drop=True)
else:
    candidateA_main = pd.DataFrame(columns=['cmdb_id','kpi_name','run_start_timestamp_utc8','run_length_minutes','max_value_in_run','threshold_p95','run_severity'])
    node_runs = pd.DataFrame(columns=candidateA_main.columns)

candidateA_table = pd.concat([candidateA_main, node_runs], ignore_index=True, sort=False).reset_index(drop=True)

# 2) Candidate B: checkoutservice-0 memory spike
if not mfr.empty:
    checkout_metric_hits = mfr[(mfr['cmdb_id']=='checkoutservice-0') & 
                               (mfr['kpi_name'].str.contains('memory_mapped', na=False) | mfr['kpi_name'].str.contains('memory_mapped_file', na=False))].copy()
    if checkout_metric_hits.empty:
        checkout_metric_hits = mfr[mfr['cmdb_id']=='checkoutservice-0'].sort_values('run_severity', ascending=False).head(5).copy()
    checkout_metric_hits = checkout_metric_hits.reset_index(drop=True)
else:
    checkout_metric_hits = pd.DataFrame(columns=mfr.columns)

# Trace rows for checkout/cart
if not tat20.empty:
    start_bound = pd.Timestamp('2022-03-20 14:00:00', tz=tz)
    end_bound   = pd.Timestamp('2022-03-20 14:30:00', tz=tz)
    trace_mask = (
        tat20['cmdb_id'].str.contains('checkoutservice', na=False) |
        tat20['cmdb_id'].str.contains('cartservice', na=False)
    )
    traces_sel = tat20[trace_mask].copy()
    traces_sel = traces_sel[
        (traces_sel['earliest_anomaly_timestamp_utc8'] >= start_bound) &
        (traces_sel['earliest_anomaly_timestamp_utc8'] <= end_bound)
    ].copy()
    traces_sel = traces_sel.loc[:, ['cmdb_id','trace_name','earliest_anomaly_timestamp_utc8','anomaly_count_in_window','max_value_in_window','threshold_p95','severity_score']]
    traces_sel = traces_sel.sort_values('severity_score', ascending=False).head(10).reset_index(drop=True)
else:
    traces_sel = pd.DataFrame(columns=['cmdb_id','trace_name','earliest_anomaly_timestamp_utc8','anomaly_count_in_window','max_value_in_window','threshold_p95','severity_score'])

# Prepare Candidate B compact table: metric hits (all metric columns) followed by trace rows (selected columns)
# For compactness, present metric rows with their main metric columns, and trace rows with requested trace columns.
candidateB_metric = checkout_metric_hits.copy()
# keep all metric columns for candidate B (as requested)
candidateB_trace = traces_sel.copy()

# 3) Error-log samples for specified cmdb_ids
targets = ["frontend-0","frontend-1","frontend-2","checkoutservice-0"]
if not err_inc.empty:
    err_sel = err_inc[err_inc['cmdb_id'].isin(targets)].copy()
    err_sel['timestamp_utc8'] = err_sel['timestamp'].apply(lambda x: pd.to_datetime(int(x), unit='s', utc=True).tz_convert(tz))
    err_sel = err_sel.sort_values(['cmdb_id','timestamp'])
    samples_list = []
    for cid in targets:
        part = err_sel[err_sel['cmdb_id']==cid].head(5).copy()
        samples_list.append(part)
    err_samples = pd.concat(samples_list, ignore_index=True, sort=False)
    err_samples = err_samples.sort_values('timestamp_utc8').loc[:, ['timestamp_utc8','cmdb_id','message']].head(10).reset_index(drop=True)
else:
    err_samples = pd.DataFrame(columns=['timestamp_utc8','cmdb_id','message'])

# Store sub-filtered tables in kernel variables
candidateA_main_filtered = candidateA_main
candidateA_node_filtered = node_runs
candidateB_metric_filtered = candidateB_metric
candidateB_trace_filtered = candidateB_trace
error_samples_filtered = err_samples

# Return only the three compact tables requested
candidateA_table, candidateB_metric, candidateB_trace, err_samples

```
Out[6]:
```
summary = (
    "Corroborating evidence for two candidate failures (incident window 2022-03-20 14:00–14:29:59 UTC+8):\n\n"
    "1) Candidate A — Node/Pod I/O & network incident:\n"
    "- Multiple high-severity runs on filesystem writes and network receive were seen on checkout/payment pods and nodes.\n"
    "- Notable items: checkoutservice-* long fs_writes runs (30 minutes starting ~14:00) with run_severity ≈ 19–20; "
    "paymentservice* short but severe container_network_receive spikes (~14:02–14:26) with severity ≈ 21.5; "
    "node-4 shows prolonged system.io.w_s run from ~14:01–14:29 (severity ≈ 12.26).\n"
    "- Interpretation: concurrent elevated disk write I/O and increased network receive on services and node(s) (node-4/node-5) — likely an I/O/network resource incident impacting checkout/payment paths.\n\n"
    "2) Candidate B — checkoutservice-0 memory spike and request latency:\n"
    "- metric: checkoutservice-0 shows a large memory-mapped-file spike at ~14:14–14:16 (max ~3,186,688; run_severity ≈ 43.22), with a secondary run later at ~14:24–14:29.\n"
    "- trace: strong, high-severity trace anomalies on checkout↔cart paths starting at 14:00 (e.g., trace.from_checkoutservice-1.duration_mean and related duration_p95 entries with severity up to ≈49), indicating sustained latency on checkout→cart calls.\n"
    "- Interpretation: a pronounced memory-mapped-file spike on checkoutservice-0 coincides with severe service-level latency in checkout↔cart traces — suggests a memory/IO-related fault in checkoutservice-0 affecting request latency.\n\n"
    "3) Error-log corroboration:\n"
    "- Frontend error messages begin at ~14:00 (many samples across frontend-0/1/2 within the first seconds of the window), matching the start of trace/metric anomalies.\n"
    "- This supports an early frontend-facing impact that aligns with backend (checkout/cart) latency and node I/O/network issues.\n\n"
    "Overall recommendation: prioritize investigation of checkoutservice-0 memory-mapped-file spike and node I/O (node-4/node-5) and check network receive patterns on payment/shipping pods. Correlate frontend error messages with trace spans from frontend→checkout→cart during 14:00–14:16 to identify root cause.\n"
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id                                           kpi_name  run_start_timestamp_utc8  run_length_minutes  max_value_in_run  threshold_p95  run_severity
0    paymentservice-1  container.node-5.container_network_receive_MB.... 2022-03-20 14:19:00+08:00                   2          0.549862       0.025181     21.836705
1   paymentservice2-0  container.node-6.container_network_receive_MB.... 2022-03-20 14:26:00+08:00                   1          0.556575       0.025657     21.693163
2    paymentservice-0  container.node-5.container_network_receive_MB.... 2022-03-20 14:15:00+08:00                   2          0.556527       0.025848     21.530359
3   checkoutservice-1      container.node-5.container_fs_writes./dev/vda 2022-03-20 14:00:00+08:00                  30         33.000000       1.650000     20.000000
4   checkoutservice-0      container.node-5.container_fs_writes./dev/vda 2022-03-20 14:00:00+08:00                  30         32.000000       1.650000     19.393939
..                ...                                                ...                       ...                 ...               ...            ...           ...
20             node-4                                      system.io.w_s 2022-03-20 14:01:00+08:00                  29          9.500000       0.775000     12.258065
21             node-5                                      system.io.w_s 2022-03-20 14:28:00+08:00                   1         84.500000      26.200000      3.225191
22             node-5                             system.net.tcp.in_segs 2022-03-20 14:13:00+08:00                   1        678.310000     213.857500      3.171785
23             node-4                                    system.cpu.user 2022-03-20 14:29:00+08:00                   1          7.810000       2.610500      2.991764
24             node-4                            system.net.tcp.out_segs 2022-03-20 14:26:00+08:00                   1       5121.730000    1812.834000      2.825261

[25 rows x 7 columns],              cmdb_id                                       kpi_name  run_start_timestamp_utc8    run_end_timestamp_utc8  run_length_minutes  max_value_in_run  min_value_in_run  threshold_p95  threshold_p5  run_severity
0  checkoutservice-0  container.node-5.container_memory_mapped_file 2022-03-20 14:14:00+08:00 2022-03-20 14:16:00+08:00                   3         3186688.0               0.0        73728.0       11673.6     43.222222
1  checkoutservice-0  container.node-5.container_memory_mapped_file 2022-03-20 14:24:00+08:00 2022-03-20 14:29:00+08:00                   6           73728.0           73728.0        73728.0       11673.6      1.000000,               cmdb_id                                  trace_name earliest_anomaly_timestamp_utc8  anomaly_count_in_window  max_value_in_window  threshold_p95  severity_score
0       cartservice-0  trace.from_checkoutservice-1.duration_mean       2022-03-20 14:00:00+08:00                       30             0.009000       0.000183       49.140049
1   checkoutservice-1        trace.to_cartservice-0.duration_mean       2022-03-20 14:00:00+08:00                       30             0.009000       0.000183       49.140049
2   checkoutservice-1         trace.to_cartservice-0.duration_p95       2022-03-20 14:00:00+08:00                       30             0.017100       0.000495       34.545455
3       cartservice-0   trace.from_checkoutservice-1.duration_p95       2022-03-20 14:00:00+08:00                       30             0.017100       0.000495       34.545455
4   checkoutservice-2        trace.to_cartservice-2.duration_mean       2022-03-20 14:00:00+08:00                       30             0.002333       0.000137       16.967273
5       cartservice-2  trace.from_checkoutservice-2.duration_mean       2022-03-20 14:00:00+08:00                       30             0.002333       0.000137       16.967273
6       cartservice-0         trace.from_frontend-2.duration_mean       2022-03-20 14:01:00+08:00                       19             0.001059       0.000067       15.712166
7       cartservice-2   trace.from_checkoutservice-2.duration_p95       2022-03-20 14:00:00+08:00                       30             0.006300       0.000467       13.475936
8   checkoutservice-2         trace.to_cartservice-2.duration_p95       2022-03-20 14:00:00+08:00                       30             0.006300       0.000467       13.475936
9  checkoutservice2-0    trace.to_currencyservice2-0.duration_p95       2022-03-20 14:10:00+08:00                        4             0.012311       0.010194        9.941667,              timestamp_utc8     cmdb_id                                            message
0 2022-03-20 14:00:00+08:00  frontend-2  severity: warning, message: failed to retrieve...
1 2022-03-20 14:00:03+08:00  frontend-0  severity: warning, message: failed to retrieve...
2 2022-03-20 14:00:03+08:00  frontend-1  severity: warning, message: failed to retrieve...
3 2022-03-20 14:00:04+08:00  frontend-2  severity: warning, message: failed to retrieve...
4 2022-03-20 14:00:05+08:00  frontend-2  severity: warning, message: failed to retrieve...
5 2022-03-20 14:00:06+08:00  frontend-0  severity: warning, message: failed to retrieve...
6 2022-03-20 14:00:06+08:00  frontend-1  severity: warning, message: failed to retrieve...
7 2022-03-20 14:00:08+08:00  frontend-0  severity: warning, message: failed to retrieve...
8 2022-03-20 14:00:10+08:00  frontend-0  severity: warning, message: failed to retrieve...
9 2022-03-20 14:00:10+08:00  frontend-1  severity: warning, message: failed to retrieve...)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```